# Analysis

Descriptive analysis of (1) the full decision corpus, (2) the training data, and (3) the model predictions and the final validated dataset.

## Corpus analysis

Volume, temporal and geographic distribution, and text-length statistics of the collected first-instance decisions.

Walk `artifacts/raw_decisions` and, for each Judilibre JSON, extract number, date, location and text. After cleaning the text, record word and character counts for both the full text and the `motivations` zone in a DataFrame.

In [ ]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import re

# Clean text (whitespace, line breaks, etc.)
def clean(text):
    return re.sub(r"\s+", " ", text.strip())

# Path to the folder containing JSON files
SOURCE_JSON_DIR = "artifacts/raw_decisions"  # not shipped — regenerated by this step (see DATA.md)

# Initialize data
data = []

# Traverse JSON files
for root, dirs, files in os.walk(SOURCE_JSON_DIR):
    for file in tqdm(files, desc="Analyzing decisions"):
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    d = json.load(f)

                number = d.get("number")
                date = d.get("decision_date")
                location = d.get("location")
                full_text = d.get("text", "")
                zones = d.get("zones", {})

                # Clean full text
                clean_full_text = clean(full_text)
                wc_full = len(clean_full_text.split())
                cc_full = len(clean_full_text)

                # "motivations" zone if present
                wc_motiv = cc_motiv = None
                motiv_zone = zones.get("motivations", [])
                if motiv_zone and isinstance(motiv_zone, list):
                    start = motiv_zone[0].get("start")
                    end = motiv_zone[0].get("end")
                    if start is not None and end is not None:
                        motiv_text = clean(full_text[start:end])
                        wc_motiv = len(motiv_text.split())
                        cc_motiv = len(motiv_text)

                # Build decision_id: number__date
                if number and date:
                    number = number.strip().replace(" ", "")
                    decision_id = f"{number}__{date[:10]}"
                else:
                    decision_id = None

                if date:
                    data.append({
                        "decision_id": decision_id,
                        "decision_date": date[:10],
                        "location": location,
                        "word_count_full": wc_full,
                        "char_count_full": cc_full,
                        "word_count_motiv": wc_motiv,
                        "char_count_motiv": cc_motiv
                    })

            except Exception as e:
                print(f"  Error in file {file}: {e}")

# Create DataFrame
df = pd.DataFrame(data)

# Descriptive statistics
temporal_counts = df["decision_date"].value_counts().sort_index()
geo_counts = df["location"].value_counts()
mean_words_full = df["word_count_full"].mean()
mean_words_motiv = df["word_count_motiv"].dropna().mean()
print(f"Total number of retrieved decisions: {len(df)}")

Drop duplicate decisions on `decision_id`.

In [ ]:
# Remove duplicates on decision_id (keep first occurrence)
df = df.drop_duplicates(subset="decision_id", keep="first")

print(f"Number of decisions after duplicate removal: {len(df)}")

Daily decision counts, with a 7-day centred rolling average and the global daily mean.

In [ ]:
# Create a copy to avoid SettingWithCopyWarning
df = df.copy()
# Convert to datetime
df['decision_date'] = pd.to_datetime(df['decision_date'])

# Deduplicate to count each decision only once per date
unique_decisions = df.drop_duplicates(subset=["decision_id"])[["decision_id", "decision_date"]]

# Count decisions per day
counts_by_day = unique_decisions['decision_date'].value_counts().sort_index()

# Rolling average
window = 7
rolling_counts = counts_by_day.rolling(window, center=True, min_periods=1).mean()

mean_daily = counts_by_day.mean()
print(f"Average decisions per day (unique): {mean_daily:.2f}")

# Plot
plt.figure(figsize=(12, 5))
plt.plot(counts_by_day.index, counts_by_day.values, label="Raw", color='lightgrey', alpha=0.5)
plt.plot(rolling_counts.index, rolling_counts.values, label=f"Rolling average ({window}d)", color='royalblue', linewidth=2)
plt.axhline(mean_daily, color='red', linestyle='--', label=f"Mean ({mean_daily:.2f})")
plt.title(f"Decisions per day ({window}-day smoothing)")
plt.xlabel("Date")
plt.ylabel("Unique decisions")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True, alpha=0.2)
plt.show()

Weekly decision counts as a bar chart, with the global weekly mean as a reference line.

In [ ]:
df = df.copy()
df['decision_date'] = pd.to_datetime(df['decision_date'])
unique_decisions = df.drop_duplicates(subset=["decision_id"])[["decision_id", "decision_date"]]

# Decisions per week
unique_decisions['week'] = unique_decisions['decision_date'].dt.to_period('W')
counts_by_week = unique_decisions['week'].value_counts().sort_index()

mean_weekly = counts_by_week.mean()
print(f"Average decisions per week (unique): {mean_weekly:.1f}")

plt.figure(figsize=(12, 5))
bars = plt.bar(counts_by_week.index.astype(str), counts_by_week.values, width=0.7, color="deepskyblue")
plt.axhline(mean_weekly, color='red', linestyle='--', label=f"Mean ({mean_weekly:.1f})")
plt.title("Decisions per week (unique)")
plt.xlabel("Week")
plt.ylabel("Unique decisions")

labels = counts_by_week.index.astype(str)
plt.xticks(
    ticks=range(len(labels)),
    labels=[label if i % 3 == 0 else "" for i, label in enumerate(labels)],
    rotation=45
)

plt.legend()
plt.tight_layout()
plt.grid(True, axis='y', alpha=0.2)
plt.show()

Map each court code to its city via `tribunal_insee_codes.xlsx`, then tabulate and plot the geographic distribution of the top 30 courts.

In [ ]:
# 1. Load court code -> city mapping
correspondance = pd.read_excel(
    "DATA/inputs/tribunal_insee_codes.xlsx"
)

dict_code_ville = dict(zip(correspondance['tj01053'], correspondance['Tribunal judiciaire de Bourg-en-Bresse']))

# 2. Create 'ville' column before any selection or stats
df["ville"] = df["location"].map(dict_code_ville)

# 3. Deduplicate decisions
unique_decisions = df.drop_duplicates(subset=["decision_id"])[["decision_id", "ville"]]

# 4. Statistics: number of cases per city (unique decisions only)
geo_counts_ville = unique_decisions["ville"].value_counts().sort_values(ascending=False)
total_decisions = geo_counts_ville.sum()

geo_table = pd.DataFrame({
    "ville": geo_counts_ville.index,
    "decision_count": geo_counts_ville.values,
    "percentage": 100 * geo_counts_ville.values / total_decisions
})
geo_table["decision_count"] = geo_table["decision_count"].apply(lambda x: f"{x:,}".replace(",", " "))
geo_table["percentage"] = geo_table["percentage"].apply(lambda x: f"{x:.2f}%")

print("\nMost frequent cities:")
print(geo_table.head(30).to_string(index=False))

# 5. Chart (top 30 cities)
top_n = 30
top_villes = geo_counts_ville.head(top_n)
top_pcts = 100 * top_villes.values / total_decisions

import matplotlib.ticker as ticker

plt.figure(figsize=(10, 6))
bars = plt.barh(top_villes.index[::-1], top_villes.values[::-1], color="cadetblue")
plt.title(f"Top {top_n} courts")
plt.xlabel("Unique decisions")
plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: format(int(x), 'n').replace(",", " ")))

for i, (bar, val, pct) in enumerate(zip(bars, top_villes.values[::-1], top_pcts[::-1])):
    label = f"{val:,.0f} ({pct:.1f} %)".replace(",", " ")
    plt.text(bar.get_width() + max(top_villes.values)*0.01,
             bar.get_y() + bar.get_height()/2,
             label,
             va='center', ha='left', fontsize=11)

plt.tight_layout()
plt.show()

Descriptive statistics (up to the 99th percentile) for full-text word and character counts.

In [ ]:
import numpy as np

# Full text statistics
desc_words = df["word_count_full"].describe(percentiles=[.25, .5, .75, .9, .95, .99])
desc_chars = df["char_count_full"].describe(percentiles=[.25, .5, .75, .9, .95, .99])

def fmt(x):
    return f"{int(x):,}".replace(",", " ")

print("\nWord count statistics (full text):")
for stat, val in desc_words.items():
    print(f"{stat:>9}: {fmt(val)}")

print("\nCharacter count statistics (full text):")
for stat, val in desc_chars.items():
    print(f"{stat:>9}: {fmt(val)}")

Cumulative distribution of word count per decision, with Q1/median/Q3/mean reference lines; the x-axis is capped at the 99.5th percentile to limit the effect of outliers.

In [ ]:
# 1. Sort values
x = np.sort(df["word_count_full"].dropna())
y = np.arange(1, len(x) + 1) / len(x)

# 2. Compute quartiles and mean
q1 = np.percentile(x, 25)
q2 = np.percentile(x, 50)
q3 = np.percentile(x, 75)
mean = np.mean(x)

# 3. Plot cumulative distribution
plt.figure(figsize=(10, 5))
plt.plot(x, y, color="crimson", linewidth=2.2, label="Cumulative (CDF)")
plt.axvline(q1, color='dodgerblue', linestyle=':', label=f"Q1 (25%) = {int(q1):,}".replace(",", " "))
plt.axvline(q2, color='violet', linestyle='--', label=f"Median (Q2) = {int(q2):,}".replace(",", " "))
plt.axvline(q3, color='orange', linestyle=':', label=f"Q3 (75%) = {int(q3):,}".replace(",", " "))
plt.axvline(mean, color='teal', linestyle='-.', label=f"Mean = {int(mean):,}".replace(",", " "))

plt.title("Cumulative distribution of word count per decision")
plt.xlabel("Word count")
plt.ylabel("Proportion of decisions (< x)")
plt.xlim(0, np.quantile(x, 0.995))
plt.ylim(0, 1.01)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Cumulative distribution of word count in the `motivations` zone, with Q1/median/Q3/mean reference lines.

In [ ]:
# 1. Sort values for motivation (excluding NaN)
x = np.sort(df["word_count_motiv"].dropna())
y = np.arange(1, len(x) + 1) / len(x)

# 2. Compute quartiles and mean
q1 = np.percentile(x, 25)
q2 = np.percentile(x, 50)
q3 = np.percentile(x, 75)
mean = np.mean(x)

# 3. Plot cumulative distribution
plt.figure(figsize=(10, 5))
plt.plot(x, y, color="seagreen", linewidth=2.2, label="Cumulative (CDF)")
plt.axvline(q1, color='dodgerblue', linestyle=':', label=f"Q1 (25%) = {int(q1):,}".replace(",", " "))
plt.axvline(q2, color='violet', linestyle='--', label=f"Median (Q2) = {int(q2):,}".replace(",", " "))
plt.axvline(q3, color='orange', linestyle=':', label=f"Q3 (75%) = {int(q3):,}".replace(",", " "))
plt.axvline(mean, color='teal', linestyle='-.', label=f"Mean = {int(mean):,}".replace(",", " "))

plt.title("Cumulative distribution of word count in motivation")
plt.xlabel("Word count (motivation)")
plt.ylabel("Proportion of decisions (< x)")
plt.xlim(0, np.quantile(x, 0.995))
plt.ylim(0, 1.01)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Cumulative distribution of character count per decision, with Q1/median/Q3/mean reference lines.

In [ ]:
# 1. Sort values
x = np.sort(df["char_count_full"].dropna())
y = np.arange(1, len(x) + 1) / len(x)

# 2. Compute quartiles and mean
q1 = np.percentile(x, 25)
q2 = np.percentile(x, 50)
q3 = np.percentile(x, 75)
mean = np.mean(x)

# 3. Plot cumulative distribution
plt.figure(figsize=(10, 5))
plt.plot(x, y, color="slateblue", linewidth=2.2, label="Cumulative (CDF)")
plt.axvline(q1, color='dodgerblue', linestyle=':', label=f"Q1 (25%) = {int(q1):,}".replace(",", " "))
plt.axvline(q2, color='violet', linestyle='--', label=f"Median (Q2) = {int(q2):,}".replace(",", " "))
plt.axvline(q3, color='orange', linestyle=':', label=f"Q3 (75%) = {int(q3):,}".replace(",", " "))
plt.axvline(mean, color='teal', linestyle='-.', label=f"Mean = {int(mean):,}".replace(",", " "))

plt.title("Cumulative distribution of character count per decision")
plt.xlabel("Character count")
plt.ylabel("Proportion of decisions (< x)")
plt.xlim(0, np.quantile(x, 0.995))
plt.ylim(0, 1.01)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Cumulative distribution of character count in the `motivations` zone, with Q1/median/Q3/mean reference lines.

In [ ]:
# Sort values for motivation (excluding NaN)
x = np.sort(df["char_count_motiv"].dropna())
y = np.arange(1, len(x) + 1) / len(x)

# Compute quartiles and mean
q1 = np.percentile(x, 25)
q2 = np.percentile(x, 50)
q3 = np.percentile(x, 75)
mean = np.mean(x)

# Plot cumulative distribution
plt.figure(figsize=(10, 5))
plt.plot(x, y, color="seagreen", linewidth=2.2, label="Cumulative (CDF)")
plt.axvline(q1, color='dodgerblue', linestyle=':', label=f"Q1 (25%) = {int(q1):,}".replace(",", " "))
plt.axvline(q2, color='violet', linestyle='--', label=f"Median (Q2) = {int(q2):,}".replace(",", " "))
plt.axvline(q3, color='orange', linestyle=':', label=f"Q3 (75%) = {int(q3):,}".replace(",", " "))
plt.axvline(mean, color='teal', linestyle='-.', label=f"Mean = {int(mean):,}".replace(",", " "))

plt.title("Cumulative distribution of character count in motivation")
plt.xlabel("Character count (motivation)")
plt.ylabel("Proportion of decisions (< x)")
plt.xlim(0, np.quantile(x, 0.995))
plt.ylim(0, 1.01)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Training-data analysis

Size of the TF-IDF training set and how the Civil Code articles used for training are distributed across the Code's books and titles.

In [ ]:
# Training set matched with TF-IDF similarity, actually used for training
import pandas as pd

# Load parquet
parquet_path = "artifacts/tfidf/tfidf_POS_chunks_train.parquet"  # not shipped — regenerated by this step (see DATA.md)
train_df = pd.read_parquet(parquet_path)

print(train_df.columns)

decision_ids_train = set(train_df["decision_id"].unique())
print(f"{len(decision_ids_train):,} unique decisions in training set")

Clean the Civil Code article list: strip the `Replier` UI artifacts left over from scraping and collapse extra whitespace, writing the result to `civil_code_article_list_cleaned.txt`.

In [ ]:
import re

input_path = "DATA/inputs/civil_code_article_list.txt"
output_path = "artifacts/reference/civil_code_article_list_cleaned.txt"  # not shipped — regenerated by this step (see DATA.md)

import os
os.makedirs("artifacts/reference", exist_ok=True)
with open(input_path, "r", encoding="utf-8") as f_in, open(output_path, "w", encoding="utf-8") as f_out:
    for line in f_in:
        # Replace 'ReplierWord' with 'Word' (keeps the rest)
        cleaned = re.sub(r"\bReplier([A-Z][a-zA-Z]*)", r"\1", line)
        # Remove standalone 'Replier'
        cleaned = re.sub(r"\bReplier\b", "", cleaned)
        # Clean double spaces
        cleaned = re.sub(r"  +", " ", cleaned).strip()
        if cleaned:
            f_out.write(cleaned + "\n")

print("Cleaned file written.")

Parse the Civil Code structure (Book -> Title -> Article) from the cleaned list and map the training-set articles onto it, counting coverage per Book and Title.

In [ ]:
import pandas as pd
import re

# 1. Load training articles
parquet_path = "artifacts/tfidf/tfidf_POS_chunks_train.parquet"  # not shipped — regenerated by this step (see DATA.md)
train_df = pd.read_parquet(parquet_path)
articles_train = set(train_df["article"].dropna().unique())
print(f"{len(articles_train):,} unique articles in training set")

# 2. Parse Civil Code structure and build ordered hierarchy
input_path = "artifacts/reference/civil_code_article_list_cleaned.txt"
structure = dict()
article2hier = dict()
total_articles = 0

current_livre = None
current_livre_label = None
current_titre = None
current_titre_label = None
has_seen_livre = False
LIVRE_PRELIM = "Titre préliminaire"

re_livre = re.compile(r'^(Livre [^\:]+):(.*)')
re_titre = re.compile(r'^(Titre [^\:]+):(.*)')
re_prelim = re.compile(r'^(Titre préliminaire)(.*)')
re_article = re.compile(r'Article ([\w-]+)')

with open(input_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # New Book?
        m_livre = re_livre.match(line)
        if m_livre:
            has_seen_livre = True
            current_livre = m_livre.group(1)
            current_livre_label = m_livre.group(2).strip()
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_train": 0
                }
            current_titre = None
            current_titre_label = None
            continue
        # Preliminary Title (only if no Book seen yet)
        m_prelim = re_prelim.match(line)
        if m_prelim and not has_seen_livre:
            current_livre = LIVRE_PRELIM
            current_livre_label = m_prelim.group(2).strip(" :")
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_train": 0
                }
            current_titre = LIVRE_PRELIM
            current_titre_label = m_prelim.group(2).strip(" :")
            if current_titre not in structure[current_livre]["titres"]:
                structure[current_livre]["titres"][current_titre] = {
                    "label": current_titre_label,
                    "count": 0,
                    "count_in_train": 0
                }
            continue
        # New Title?
        m_titre = re_titre.match(line)
        if m_titre:
            current_titre = m_titre.group(1)
            current_titre_label = m_titre.group(2).strip()
            if current_livre is None:
                current_livre = LIVRE_PRELIM
                current_livre_label = ""
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_train": 0
                }
            if current_titre not in structure[current_livre]["titres"]:
                structure[current_livre]["titres"][current_titre] = {
                    "label": current_titre_label,
                    "count": 0,
                    "count_in_train": 0
                }
            continue
        # Articles on the line
        if "Article" in line:
            articles = re_article.findall(line)
            n = len(articles)
            if n:
                total_articles += n
                if current_titre and current_livre:
                    structure[current_livre]["titres"][current_titre]["count"] += n
                if current_livre:
                    structure[current_livre]["count"] += n
                for art in articles:
                    article2hier[art] = (current_livre, current_titre)

# 3. Count how many training articles are covered per Book/Title
for art in articles_train:
    art_clean = art.replace("Article ", "").strip()
    if art_clean in article2hier:
        livre, titre = article2hier[art_clean]
        structure[livre]["count_in_train"] += 1
        if titre and titre in structure[livre]["titres"]:
            structure[livre]["titres"][titre]["count_in_train"] += 1

total_used = sum([structure[livre]["count_in_train"] for livre in structure])

print(f"\nArticles used in TRAIN found in structure: {total_used}\n")
print("Distribution Book -> Title (in training set)\n")
for livre, ldata in structure.items():
    livre_label = ldata["label"]
    lcount = ldata["count_in_train"]
    lpct = 100 * lcount / total_used if total_used else 0
    print(f"  {livre} ({livre_label}) : {lcount:>4d} articles ({lpct:>5.2f} %)")
    for titre, tdata in ldata["titres"].items():
        tlabel = tdata["label"]
        tcount = tdata["count_in_train"]
        tpct = 100 * tcount / total_used if total_used else 0
        print(f"    {titre:<15s} ({tlabel:<45s}) : {tcount:>4d} articles ({tpct:>5.2f} %)")
    print()

Civil Code coverage by the training set: per Book and Title, total articles versus articles present in training.

In [ ]:
total_articles = sum([ldata["count"] for ldata in structure.values()])
total_train = sum([ldata.get("count_in_train", 0) for ldata in structure.values()])

print(f"\n{'Book / Title':<75s} | Civil Code | Train | Coverage")
print("-" * 100)

for livre, ldata in structure.items():
    livre_label = ldata["label"]
    n_total = ldata["count"]
    n_train = ldata.get("count_in_train", 0)
    pct_total = 100 * n_total / total_articles if total_articles else 0
    pct_train = 100 * n_train / total_train if total_train else 0
    couverture = 100 * n_train / n_total if n_total else 0

    print(f"  {livre:<20s} ({livre_label:<45s}) | {n_total:>5d} ({pct_total:5.2f}%) | {n_train:>4d} ({pct_train:5.2f}%) | {couverture:5.2f}%")

    for titre, tdata in ldata["titres"].items():
        tlabel = tdata["label"]
        tcount = tdata["count"]
        ttrain = tdata.get("count_in_train", 0)
        pct_titre = 100 * tcount / total_articles if total_articles else 0
        pct_train_titre = 100 * ttrain / total_train if total_train else 0
        couverture_titre = 100 * ttrain / tcount if tcount else 0
        print(f"    {titre:<15s} ({tlabel:<40s}) | {tcount:>5d} ({pct_titre:5.2f}%) | {ttrain:>4d} ({pct_train_titre:5.2f}%) | {couverture_titre:5.2f}%")
    print()

## Predictions analysis

Coverage of the Civil Code by the test-set predictions, and the distribution of the final o3-validated dataset across Civil Code books.

Coverage of the Civil Code by the test-set predictions: parse the Book -> Title -> Article structure and map the predicted articles (`pred_art`) onto it. The next cell prints two tables — first the raw distribution of predicted articles, then the full coverage breakdown per Book and Title.

In [ ]:
import pandas as pd
import re

# 1. Load unique predicted articles from the inference output
parquet_path = "artifacts/inference/output_predictions_unique.parquet"  # not shipped — regenerated by this step (see DATA.md)
df_pred = pd.read_parquet(parquet_path)
articles_test = set(df_pred["pred_art"].dropna().astype(str).str.replace("Article ", "").str.strip().unique())
print(f"{len(articles_test):,} unique articles in TEST set")

# 2. Parse Civil Code structure
input_path = "artifacts/reference/civil_code_article_list_cleaned.txt"
structure = dict()
article2hier = dict()
total_articles = 0

current_livre = None
current_livre_label = None
current_titre = None
current_titre_label = None
has_seen_livre = False
LIVRE_PRELIM = "Titre préliminaire"

re_livre = re.compile(r'^(Livre [^\:]+):(.*)')
re_titre = re.compile(r'^(Titre [^\:]+):(.*)')
re_prelim = re.compile(r'^(Titre préliminaire)(.*)')
re_article = re.compile(r'Article ([\w-]+)')

with open(input_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # New Book?
        m_livre = re_livre.match(line)
        if m_livre:
            has_seen_livre = True
            current_livre = m_livre.group(1)
            current_livre_label = m_livre.group(2).strip()
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_test": 0
                }
            current_titre = None
            current_titre_label = None
            continue
        # Preliminary Title (outside any Book)
        m_prelim = re_prelim.match(line)
        if m_prelim and not has_seen_livre:
            current_livre = LIVRE_PRELIM
            current_livre_label = m_prelim.group(2).strip(" :")
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_test": 0
                }
            current_titre = LIVRE_PRELIM
            current_titre_label = m_prelim.group(2).strip(" :")
            if current_titre not in structure[current_livre]["titres"]:
                structure[current_livre]["titres"][current_titre] = {
                    "label": current_titre_label,
                    "count": 0,
                    "count_in_test": 0
                }
            continue
        # New Title?
        m_titre = re_titre.match(line)
        if m_titre:
            current_titre = m_titre.group(1)
            current_titre_label = m_titre.group(2).strip()
            if current_livre is None:
                current_livre = LIVRE_PRELIM
                current_livre_label = ""
            if current_livre not in structure:
                structure[current_livre] = {
                    "label": current_livre_label,
                    "titres": dict(),
                    "count": 0,
                    "count_in_test": 0
                }
            if current_titre not in structure[current_livre]["titres"]:
                structure[current_livre]["titres"][current_titre] = {
                    "label": current_titre_label,
                    "count": 0,
                    "count_in_test": 0
                }
            continue
        # Articles on the line
        if "Article" in line:
            articles = re_article.findall(line)
            n = len(articles)
            if n:
                total_articles += n
                if current_titre and current_livre:
                    structure[current_livre]["titres"][current_titre]["count"] += n
                if current_livre:
                    structure[current_livre]["count"] += n
                for art in articles:
                    article2hier[art] = (current_livre, current_titre)

# 3. Count how many test articles are covered per Book/Title
for art in articles_test:
    art_clean = art.replace("Article ", "").strip()
    if art_clean in article2hier:
        livre, titre = article2hier[art_clean]
        structure[livre].setdefault("count_in_test", 0)
        structure[livre]["count_in_test"] += 1
        if titre and titre in structure[livre]["titres"]:
            structure[livre]["titres"][titre].setdefault("count_in_test", 0)
            structure[livre]["titres"][titre]["count_in_test"] += 1

# Summary display
total_used_test = sum([structure[livre].get("count_in_test", 0) for livre in structure])

print(f"\nTest articles found in structure: {total_used_test}\n")
print("Distribution Book -> Title (in TEST set)\n")
for livre, ldata in structure.items():
    livre_label = ldata["label"]
    lcount = ldata.get("count_in_test", 0)
    lpct = 100 * lcount / total_used_test if total_used_test else 0
    print(f"  {livre} ({livre_label}) : {lcount:>4d} articles ({lpct:>5.2f} %)")
    for titre, tdata in ldata["titres"].items():
        tlabel = tdata["label"]
        tcount = tdata.get("count_in_test", 0)
        tpct = 100 * tcount / total_used_test if total_used_test else 0
        print(f"    {titre:<15s} ({tlabel:<45s}) : {tcount:>4d} articles ({tpct:>5.2f} %)")
    print()


# Coverage per Book and Title (percentage of Civil Code and test set)
total_articles = sum([ldata["count"] for ldata in structure.values()])
total_test = sum([ldata.get("count_in_test", 0) for ldata in structure.values()])

print(f"\n{'Book / Title':<75s} | Civil Code | Test | Coverage")
print("-" * 100)

for livre, ldata in structure.items():
    livre_label = ldata["label"]
    n_total = ldata["count"]
    n_test = ldata.get("count_in_test", 0)
    pct_total = 100 * n_total / total_articles if total_articles else 0
    pct_test = 100 * n_test / total_test if total_test else 0
    coverage = 100 * n_test / n_total if n_total else 0

    print(f"  {livre:<20s} ({livre_label:<45s}) | {n_total:>5d} ({pct_total:5.2f}%) | {n_test:>4d} ({pct_test:5.2f}%) | {coverage:5.2f}%")

    for titre, tdata in ldata["titres"].items():
        tlabel = tdata["label"]
        tcount = tdata["count"]
        ttest = tdata.get("count_in_test", 0)
        pct_titre = 100 * tcount / total_articles if total_articles else 0
        pct_test_titre = 100 * ttest / total_test if total_test else 0
        coverage_titre = 100 * ttest / tcount if tcount else 0
        print(f"    {titre:<15s} ({tlabel:<40s}) | {tcount:>5d} ({pct_titre:5.2f}%) | {ttest:>4d} ({pct_test_titre:5.2f}%) | {coverage_titre:5.2f}%")
    print()

Distribution of the final o3-validated dataset across Civil Code books, printed as a summary and as the LaTeX table used in the paper.

In [ ]:
import pandas as pd
import re

# Path to file
excel_path = "artifacts/validation/full_preds_o3_final.xlsx"  # final annotated dataset from step 07; not shipped - see DATA.md
df = pd.read_excel(excel_path)

def get_livre(art_str):
    """Returns the Civil Code book based on article number"""
    match = re.match(r'(\d+)', str(art_str))
    if not match:
        return None
    num = int(match.group(1))

    if num <= 6:
        return "Titre préliminaire"
    elif num <= 515:
        return "Livre I: Des personnes"
    elif num <= 710:
        return "Livre II: Des biens"
    elif num <= 2278:
        return "Livre III: Des différentes manières dont on acquiert la propriété"
    elif num <= 2488:
        return "Livre IV: Des sûretés"
    else:
        return "Livre V: Dispositions applicables à Mayotte"

df['livre'] = df['pred_art'].apply(get_livre)

# Statistics
pairs_by_livre = df.groupby('livre').size()
articles_by_livre = df.groupby('livre')['pred_art'].nunique()
total_pairs = len(df)
total_articles = df['pred_art'].nunique()

print("=" * 70)
print("STATISTICS BY CIVIL CODE BOOK")
print("=" * 70)
print(f"\nTotal pairs: {total_pairs} | Unique articles: {total_articles}\n")

for livre in pairs_by_livre.index:
    n_pairs = pairs_by_livre[livre]
    pct_pairs = 100 * n_pairs / total_pairs
    n_arts = articles_by_livre[livre]
    pct_arts = 100 * n_arts / total_articles
    print(f"{livre:<65s}")
    print(f"  Pairs: {n_pairs:>4d} ({pct_pairs:>5.1f}%) | Articles: {n_arts:>4d} ({pct_arts:>5.1f}%)")

# LaTeX table
print("\n\n% --- LATEX TABLE ---")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\begin{tabular}{lrrrr}")
print(r"\toprule")
print(r"\textbf{Book} & \textbf{Pairs} & \textbf{\%} & \textbf{Articles} & \textbf{\%} \\")
print(r"\midrule")

for livre in pairs_by_livre.index:
    n_pairs = pairs_by_livre[livre]
    pct_pairs = 100 * n_pairs / total_pairs
    n_arts = articles_by_livre[livre]
    pct_arts = 100 * n_arts / total_articles

    livre_short = livre.replace("Livre ", "Book ").split(":")[0]
    if "préliminaire" in livre:
        livre_short = "Preliminary Title"

    print(f"{livre_short:<25s} & {n_pairs:>4d} & {pct_pairs:>5.1f}\\% & {n_arts:>4d} & {pct_arts:>5.1f}\\% \\\\")

print(r"\midrule")
print(f"{'Total':<25s} & {total_pairs:>4d} & 100.0\\% & {total_articles:>4d} & 100.0\\% \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{Distribution of the dataset across Civil Code books.}")
print(r"\label{tab:book_distribution}")
print(r"\end{table}")